# 07 · One diverse virtual-site box — an adversarial pathway-equivalence test

This notebook packs **one** box with all of notebook 06's diverse virtual-site
solutes (every SMIRNOFF vsite type) in **mostly OPC water**, and parameterizes it
**both ways** — native Interchange and `openmmforcefields` — across **three charge
methods**, with the deck stacked to expose any order- or index-driven mistake in
the export:

* **Two copies of every solute**, placed **noncontiguously** in the topology (the
two copies are separated by the whole water block), and **one copy has its atoms
reversed** (`Molecule.remap`). A correct export must not care about atom order or
where a molecule sits in the topology.
* The virtual-site definitions are **written inline below**, with **every**
`charge_increment` and Lennard-Jones (`sigma`/`epsilon`) value **unique and
nonzero** — so every vsite is individually distinguishable and any mis-assignment
shows up as a wrong energy.
* The whole build + simulate is run for **three charge methods**: `NAGLCharges` and
`ToolkitAM1BCC` defined in the force field, and **user charges** injected via
`charge_from_molecules` (a separate code path) — the latter also stressing that
preset charges map correctly onto the reversed-order copies.

The headline result is that, **for each charge method, Interchange and
`openmmforcefields` agree on the single-point energy to machine precision**. More
hard assertions: the reversed-order copy of each solute gets the *same* parameters
as the in-order copy, every box is net-neutral, and every system runs stable NPT
dynamics. A topology + trajectory are written per (charge method × pathway) and
temperature/density are reported.

In [1]:
import copy
import io
import os
import warnings

import numpy as np
import openmm
import openmm.unit as u
from openff.toolkit import ForceField as OFF
from openff.toolkit import Molecule
from openff.units import unit as offunit
from openff.interchange.components._packmol import pack_box, UNIT_CUBE
from openmm import LangevinMiddleIntegrator, MonteCarloBarostat
from openmm.app import ForceField as OMMForceField
from openmm.app import Modeller, PME, PDBFile, DCDFile
from openmmforcefields.generators import SMIRNOFFTemplateGenerator

warnings.filterwarnings("ignore")
KB = u.MOLAR_GAS_CONSTANT_R
KCAL = u.kilocalories_per_mole

# Simulation settings (overridable via env for a fast dry run).
CUTOFF_NM, SWITCH_NM, EWALD_TOL, HMR_AMU, TEMP = 0.9, 0.8, 1.0e-5, 3.024, 300
DT_FS = float(os.environ.get("NB07_DT_FS", "4.0"))
N_WATER = int(os.environ.get("NB07_N_WATER", "700"))       # "mostly water"
EQUIL_NS = float(os.environ.get("NB07_EQUIL_NS", "0.05"))
PROD_NS = float(os.environ.get("NB07_PROD_NS", "0.5"))
N_FRAMES = int(os.environ.get("NB07_N_FRAMES", "100"))
EQUIL_STEPS = max(1, round(EQUIL_NS * 1e6 / DT_FS))
PROD_STEPS = max(N_FRAMES, round(PROD_NS * 1e6 / DT_FS))
TRAJ_DIR = "traj_07"
os.makedirs(TRAJ_DIR, exist_ok=True)

_AVAILABLE = {openmm.Platform.getPlatform(i).getName()
              for i in range(openmm.Platform.getNumPlatforms())}
PLATFORM = next((p for p in ("CUDA", "OpenCL", "CPU") if p in _AVAILABLE), "Reference")
print(f"platform={PLATFORM}  dt={DT_FS} fs (HMR {HMR_AMU} amu)  N_water={N_WATER}  "
      f"equil={EQUIL_STEPS} steps  prod={PROD_STEPS} steps  cutoff={CUTOFF_NM} nm")

platform=OpenCL  dt=4.0 fs (HMR 3.024 amu)  N_water=700  equil=12500 steps  prod=125000 steps  cutoff=0.9 nm


## Force field with the virtual-site definitions written inline

Sage 2.3 + OPC water, then six virtual-site parameters added directly here —
carbonyl `MonovalentLonePair`, aromatic-N `DivalentLonePair`, amine
`TrivalentLonePair`, and `BondCharge` on aromatic C–Cl, aromatic C–Br and **C–F**.
Every `charge_increment`, `sigma` and `epsilon` is a distinct nonzero value. The
base charge model is `NAGLCharges`; a second copy swaps that handler for
`ToolkitAM1BCC`. Both are written to file so the `openmmforcefields` pathway
consumes exactly these parameters.

In [2]:
e, A, deg = offunit.elementary_charge, offunit.angstrom, offunit.degree
kcal = offunit.kilocalorie / offunit.mole
COMBO_OFFXML = "vsite_ff_combined.offxml"
COMBO_AM1_OFFXML = "vsite_ff_combined_am1bcc.offxml"

VSITE_FF = OFF("openff-2.3.0.offxml", "opc.offxml")
_vs = VSITE_FF.get_parameter_handler("VirtualSites")
for _p in [
    dict(smirks="[#8:1]=[#6:2]-[#6:3]", type="MonovalentLonePair", name="EP",
         match="all_permutations", distance=0.30 * A, outOfPlaneAngle=0.0 * deg,
         inPlaneAngle=120.0 * deg, charge_increment1=0.021 * e,
         charge_increment2=0.023 * e, charge_increment3=0.027 * e,
         sigma=1.40 * A, epsilon=0.040 * kcal),
    dict(smirks="[#6X3H1a:2]1:[#7X2a:1]:[#6X3H1a:3]:[#6X3a]:[#6X3a]:[#6X3a]1",
         type="DivalentLonePair", name="EP", match="once", distance=0.30 * A,
         outOfPlaneAngle=0.0 * deg, charge_increment1=0.031 * e,
         charge_increment2=0.033 * e, charge_increment3=0.037 * e,
         sigma=1.50 * A, epsilon=0.050 * kcal),
    dict(smirks="[#6:2]~[#7X3:1](~[#6:4])~[#6:3]", type="TrivalentLonePair", name="EP",
         match="once", distance=0.30 * A, charge_increment1=0.041 * e,
         charge_increment2=0.043 * e, charge_increment3=0.047 * e,
         charge_increment4=0.049 * e, sigma=1.60 * A, epsilon=0.060 * kcal),
    dict(smirks="[#6a:2]-[#17:1]", type="BondCharge", name="EP",
         match="all_permutations", distance=1.45 * A, charge_increment1=-0.055 * e,
         charge_increment2=0.011 * e, sigma=1.10 * A, epsilon=0.010 * kcal),
    dict(smirks="[#6a:2]-[#35:1]", type="BondCharge", name="EP",
         match="all_permutations", distance=1.55 * A, charge_increment1=-0.077 * e,
         charge_increment2=0.013 * e, sigma=1.20 * A, epsilon=0.020 * kcal),
    dict(smirks="[#6:2]-[#9:1]", type="BondCharge", name="EP",
         match="all_permutations", distance=1.40 * A, charge_increment1=-0.093 * e,
         charge_increment2=0.017 * e, sigma=1.30 * A, epsilon=0.030 * kcal),
]:
    _vs.add_parameter(_p)
VSITE_FF.to_file(COMBO_OFFXML)

# Same force field, but with ToolkitAM1BCC as the base charge model instead of NAGL.
VSITE_FF_AM1 = copy.deepcopy(VSITE_FF)
VSITE_FF_AM1.deregister_parameter_handler("NAGLCharges")
VSITE_FF_AM1.get_parameter_handler("ToolkitAM1BCC", {"version": "0.3"})
VSITE_FF_AM1.to_file(COMBO_AM1_OFFXML)
print(f"virtual-site parameters written inline: {len(_vs.parameters)} "
      "(Monovalent, Divalent, Trivalent, BondCharge Cl/Br/F)")

virtual-site parameters written inline: 7 (Monovalent, Divalent, Trivalent, BondCharge Cl/Br/F)


## Solutes: two copies each, one with reversed atom order

All four vsite types plus multi-vsite molecules. Two of them carry **two
`BondCharge` vsites of the same type with different parameters**: 1-bromo-4-
chlorobenzene (C–Br + C–Cl) and 1-chloro-4-fluorobenzene (C–F + C–Cl, which also
exercises the fluorine vsite). Copy **A** keeps the toolkit's atom order; copy
**B** is the same molecule with its atoms reversed via `Molecule.remap`.

In [3]:
SMILES = [
    "CC(=O)OC",             # methyl acetate         MonovalentLonePair
    "CC(C)=O",              # acetone                MonovalentLonePair
    "NC(=O)c1ccccc1",       # benzamide              MonovalentLonePair
    "Cc1cncc(C)c1",         # 3,5-lutidine           DivalentLonePair
    "c1ccncc1",             # pyridine               DivalentLonePair
    "Clc1ccc(Br)cc1",       # 1-bromo-4-chlorobenzene   BondCharge Cl + Br
    "c1ccc(Cl)cc1",         # chlorobenzene          BondCharge Cl
    "Fc1ccc(Cl)cc1",        # 1-chloro-4-fluorobenzene  BondCharge F + Cl
    "CN(C)C",               # trimethylamine         TrivalentLonePair
    "CN(C)c1ccccc1",        # N,N-dimethylaniline    TrivalentLonePair
    "CC(=O)c1ccc(Cl)cc1",   # 4'-chloroacetophenone  Monovalent + BondCharge
    "Brc1cccnc1",           # 3-bromopyridine        Divalent + BondCharge
]

solutesA = [Molecule.from_smiles(s, allow_undefined_stereo=True) for s in SMILES]
solutesB = [m.remap({i: m.n_atoms - 1 - i for i in range(m.n_atoms)},
                    current_to_new=True) for m in solutesA]   # atoms reversed
water = Molecule.from_smiles("O")

## Check 1 — reversed-order copy gets the same parameters, and each solute is neutral

For every solute, build an isolated system for copy A and for the reversed copy B
and compare the **multiset** of per-particle `(charge, sigma, epsilon)` over all
atoms *and* virtual sites. Order-invariant assignment ⇒ the multisets match (this
also proves both copies get the same vsite count). Summing the charges confirms
each molecule stays net-neutral even with the unique nonzero charge increments.

In [4]:
for smi, mA, mB in zip(SMILES, solutesA, solutesB):
    sigs = []
    for m in (mA, mB):
        nb = next(f for f in VSITE_FF.create_openmm_system(m.to_topology()).getForces()
                  if isinstance(f, openmm.NonbondedForce))
        sigs.append(sorted(
            (round(nb.getParticleParameters(i)[0].value_in_unit(e), 6),
             round(nb.getParticleParameters(i)[1].value_in_unit(A), 6),
             round(nb.getParticleParameters(i)[2].value_in_unit(kcal), 6))
            for i in range(nb.getNumParticles())))
    net = sum(p[0] for p in sigs[0])
    assert sigs[0] == sigs[1], f"{smi}: reversed-order copy got different parameters"
    assert abs(net) < 1e-6, f"{smi}: net charge {net:.2e} e (should be neutral)"
print(f"reversed-order copies match the in-order copies, and all {len(SMILES)} solutes "
      "are net-neutral, despite the unique nonzero vsite increments")

reversed-order copies match the in-order copies, and all 12 solutes are net-neutral, despite the unique nonzero vsite increments


## Pack one box, with the two copies of each solute scattered apart

`pack_box` preserves the input order, so packing `[A0..A11, water×N, B0..B11]` puts
copy A of solute *i* near the front and copy B after the entire water block — the
two copies are never adjacent.

In [5]:
MOLECULES = solutesA + [water] + solutesB
COUNTS = [1] * len(solutesA) + [N_WATER] + [1] * len(solutesB)
off_top = pack_box(molecules=MOLECULES, number_of_copies=COUNTS,
                   target_density=950.0 * offunit.kilogram / offunit.meter ** 3,
                   box_shape=UNIT_CUBE)
positions_nm = np.asarray(off_top.get_positions().m_as(offunit.nanometer))
box = np.asarray(off_top.box_vectors.m_as(offunit.nanometer))
L = float(np.min(np.diag(box)))

formulas = [m.to_hill_formula() for m in off_top.molecules]      # no conformer round-trip
copy_slots = [i for i, f in enumerate(formulas) if f == solutesA[0].to_hill_formula()]
print(f"packed {2 * len(solutesA)} solute copies + {N_WATER} OPC waters "
      f"({off_top.n_atoms} real atoms) into a {L:.2f} nm box")
print(f"the two copies of '{SMILES[0]}' sit at molecule indices {copy_slots} "
      f"of {off_top.n_molecules} (separated by the water block)")
assert L > 2 * CUTOFF_NM, f"box {L:.2f} nm too small for {CUTOFF_NM} nm cutoff"
assert len(copy_slots) == 2 and abs(copy_slots[0] - copy_slots[1]) > 1, "copies are contiguous"

packed 24 solute copies + 700 OPC waters (2424 real atoms) into a 2.99 nm box
the two copies of 'CC(=O)OC' sit at molecule indices [0, 712] of 724 (separated by the water block)


## The three charge methods

`NAGLCharges` and `ToolkitAM1BCC` come from the two force fields above. **User
charges** are AM1-BCC values computed *outside* the force field and injected with
`charge_from_molecules` — a separate code path (their values happen to equal
`ToolkitAM1BCC` because both are AM1-BCC, but the mechanism is distinct). Each
entry is `(label, key, force_field, offxml, generator_molecules, user_charges)`.

In [6]:
preset = [Molecule.from_smiles(s, allow_undefined_stereo=True) for s in SMILES]
for m in preset:
    m.assign_partial_charges("am1bcc")     # user-supplied charges, computed outside the FF

CHARGE_METHODS = [
    ("NAGLCharges (force field)", "nagl", VSITE_FF, COMBO_OFFXML, MOLECULES, None),
    ("ToolkitAM1BCC (force field)", "am1bcc", VSITE_FF_AM1, COMBO_AM1_OFFXML, MOLECULES, None),
    ("user charges (preset AM1-BCC)", "user", VSITE_FF, COMBO_OFFXML, preset + [water], preset),
]

## Build both pathways for each charge method, then simulate

`simulate` minimises → thermalises → runs NPT, writing a topology PDB + DCD and
returning mean temperature and density (± std). For each charge method we build the
box through Interchange and through `openmmforcefields`, harmonise the PME settings,
assert the two pathways agree at the packed coordinates and that the box is neutral,
then run dynamics through each.

In [7]:
def simulate(name, system, positions, app_top):
    sim = openmm.XmlSerializer.deserialize(openmm.XmlSerializer.serialize(system))
    sim.addForce(MonteCarloBarostat(1.0 * u.bar, TEMP * u.kelvin, 25))
    integ = LangevinMiddleIntegrator(TEMP * u.kelvin, 1.0 / u.picosecond, DT_FS * u.femtosecond)
    ctx = openmm.Context(sim, integ, openmm.Platform.getPlatformByName(PLATFORM))
    ctx.setPeriodicBoxVectors(*(box * u.nanometer))
    ctx.setPositions(np.nan_to_num(np.asarray(positions.value_in_unit(u.nanometer))) * u.nanometer)
    ctx.computeVirtualSites()
    openmm.LocalEnergyMinimizer.minimize(ctx, maxIterations=1000)
    ctx.setVelocitiesToTemperature(TEMP * u.kelvin)
    integ.step(EQUIL_STEPS)

    massive = sum(sim.getParticleMass(i).value_in_unit(u.dalton) > 0 for i in range(sim.getNumParticles()))
    has_cmm = any(isinstance(f, openmm.CMMotionRemover) for f in sim.getForces())
    ndof = 3 * massive - sim.getNumConstraints() - (3 if has_cmm else 0)
    total_mass = sum(sim.getParticleMass(i).value_in_unit(u.dalton) for i in range(sim.getNumParticles()))

    pdb_path = os.path.join(TRAJ_DIR, f"diverse_{name}.pdb")
    dcd_path = os.path.join(TRAJ_DIR, f"diverse_{name}.dcd")
    app_top.setPeriodicBoxVectors(ctx.getState().getPeriodicBoxVectors())
    with open(pdb_path, "w") as fh:
        PDBFile.writeFile(app_top, ctx.getState(getPositions=True, enforcePeriodicBox=True).getPositions(), fh)
    dcd_file = open(dcd_path, "wb")
    dcd = DCDFile(dcd_file, app_top, DT_FS * u.femtosecond)

    stride = max(1, PROD_STEPS // N_FRAMES)
    temps, densities = [], []
    for _ in range(N_FRAMES):
        integ.step(stride)
        s = ctx.getState(getEnergy=True, getPositions=True, enforcePeriodicBox=True)
        temps.append((2 * s.getKineticEnergy() / (ndof * KB)).value_in_unit(u.kelvin))
        vol = np.linalg.det(s.getPeriodicBoxVectors(asNumpy=True).value_in_unit(u.nanometer))
        densities.append((total_mass / 6.02214076e23) / (vol * 1e-21))   # g/cm^3
        dcd.writeModel(s.getPositions(), periodicBoxVectors=s.getPeriodicBoxVectors())
    dcd_file.close()

    finite = bool(np.all(np.isfinite(
        ctx.getState(getPositions=True).getPositions(asNumpy=True).value_in_unit(u.nanometer))))
    del ctx, integ
    assert finite, f"{name}: non-finite dynamics"
    assert 250 < np.mean(temps) < 350, f"{name}: temperature {np.mean(temps):.1f} off setpoint"
    assert 0.7 < np.mean(densities) < 1.3, f"{name}: unphysical density {np.mean(densities):.3f}"
    return float(np.mean(temps)), float(np.std(temps)), float(np.mean(densities)), float(np.std(densities))


report, parity = {}, {}
for label, key, ff, offxml, gen_mols, user_charges in CHARGE_METHODS:
    # --- Interchange pathway ---
    interchange = ff.create_interchange(off_top, charge_from_molecules=user_charges)
    ic_sys = interchange.to_openmm_system(hydrogen_mass=HMR_AMU)
    ic_pos = np.zeros((ic_sys.getNumParticles(), 3))
    ic_pos[:len(positions_nm)] = positions_nm
    built = {"interchange": (ic_sys, ic_pos * u.nanometer, interchange.to_openmm_topology())}

    # --- openmmforcefields pathway ---
    generator = SMIRNOFFTemplateGenerator(molecules=gen_mols, forcefield=offxml)
    omm_ff = OMMForceField()
    omm_ff.registerTemplateGenerator(generator.generator)
    loaded = []
    for mol in gen_mols:                       # dedupe isomorphic residue templates
        if any(mol.is_isomorphic_with(seen) for seen in loaded):
            continue
        omm_ff.loadFile(io.StringIO(generator.generate_residue_template(mol)))
        loaded.append(mol)
    modeller = Modeller(off_top.to_openmm(), positions_nm * u.nanometer)
    modeller.addExtraParticles(omm_ff)
    om_sys = omm_ff.createSystem(modeller.topology, nonbondedMethod=PME,
                                 nonbondedCutoff=CUTOFF_NM * u.nanometer,
                                 hydrogenMass=HMR_AMU * u.dalton)
    om_pos = np.nan_to_num(np.asarray(modeller.positions.value_in_unit(u.nanometer))) * u.nanometer
    built["openmmforcefields"] = (om_sys, om_pos, modeller.topology)

    for sysm, _, _ in built.values():          # harmonise PME on both
        for f in sysm.getForces():
            if isinstance(f, openmm.NonbondedForce):
                f.setNonbondedMethod(openmm.NonbondedForce.PME)
                f.setCutoffDistance(CUTOFF_NM * u.nanometer)
                f.setUseSwitchingFunction(True)
                f.setSwitchingDistance(SWITCH_NM * u.nanometer)
                f.setEwaldErrorTolerance(EWALD_TOL)

    # single-point pathway parity + box neutrality (inline)
    e_sp = {}
    for pathway, (sysm, pos, _) in built.items():
        integ = openmm.VerletIntegrator(1.0 * u.femtosecond)
        ctx = openmm.Context(sysm, integ, openmm.Platform.getPlatformByName(PLATFORM))
        ctx.setPeriodicBoxVectors(*(box * u.nanometer))
        ctx.setPositions(np.nan_to_num(np.asarray(pos.value_in_unit(u.nanometer))) * u.nanometer)
        ctx.computeVirtualSites()
        e_sp[pathway] = ctx.getState(getEnergy=True).getPotentialEnergy().value_in_unit(KCAL)
        del ctx, integ
    rel = abs(e_sp["interchange"] - e_sp["openmmforcefields"]) / abs(e_sp["interchange"])
    parity[key] = rel
    nbf = next(f for f in ic_sys.getForces() if isinstance(f, openmm.NonbondedForce))
    totq = sum(nbf.getParticleParameters(i)[0].value_in_unit(u.elementary_charge)
               for i in range(nbf.getNumParticles()))
    print(f"\n[{label}] IC={e_sp['interchange']:.3f}  OMMFFS={e_sp['openmmforcefields']:.3f} "
          f"kcal/mol  rel.diff={rel:.2e}  total charge={totq:+.1e} e")
    assert rel < 1e-4, f"{label}: pathway energies disagree (rel {rel:.2e})"
    assert abs(totq) < 1e-6, f"{label}: box not neutral ({totq:.2e} e)"

    # dynamics through each pathway
    report[key] = {}
    for pathway, (system, pos, top) in built.items():
        report[key][pathway] = simulate(f"{key}_{pathway}", system, pos, top)
        Tm, Ts, Dm, Ds = report[key][pathway]
        print(f"  [{pathway:17s}] T={Tm:.1f}+/-{Ts:.1f} K  density={Dm:.3f}+/-{Ds:.3f} g/cm^3")


[NAGLCharges (force field)] IC=14401.137  OMMFFS=14401.135 kcal/mol  rel.diff=1.34e-07  total charge=+3.1e-16 e
  [interchange      ] T=302.7+/-6.5 K  density=1.025+/-0.008 g/cm^3
  [openmmforcefields] T=301.1+/-5.6 K  density=1.024+/-0.008 g/cm^3

[ToolkitAM1BCC (force field)] IC=14389.656  OMMFFS=14389.649 kcal/mol  rel.diff=4.78e-07  total charge=+6.6e-17 e
  [interchange      ] T=300.8+/-6.0 K  density=1.025+/-0.008 g/cm^3
  [openmmforcefields] T=299.8+/-5.6 K  density=1.025+/-0.009 g/cm^3

[user charges (preset AM1-BCC)] IC=14389.653  OMMFFS=14389.651 kcal/mol  rel.diff=1.62e-07  total charge=+6.6e-17 e
  [interchange      ] T=301.2+/-5.9 K  density=1.025+/-0.009 g/cm^3
  [openmmforcefields] T=301.3+/-5.6 K  density=1.025+/-0.007 g/cm^3


## Summary

In [8]:
print(f"{'charge method':30s} {'pathway':18s} {'T (K)':>15s} {'density (g/cm^3)':>19s} {'parity':>10s}")
print("-" * 96)
for label, key, *_ in CHARGE_METHODS:
    for j, pathway in enumerate(("interchange", "openmmforcefields")):
        Tm, Ts, Dm, Ds = report[key][pathway]
        print(f"{label if j == 0 else '':30s} {pathway:18s} {Tm:6.1f} +/- {Ts:4.1f} "
              f"{Dm:10.3f} +/- {Ds:5.3f} {(f'{parity[key]:.1e}' if j == 0 else ''):>10s}")
print("\nNotebook 07 complete: one box of all of notebook 06's diverse solutes — two copies "
      "each, scattered apart, one copy atom-reversed, with unique nonzero inline vsite "
      "parameters — was parameterized through both Interchange and openmmforcefields across "
      "three charge methods (NAGLCharges, ToolkitAM1BCC, and user charges). The reversed copies "
      "matched the originals, every box stayed neutral, the two pathways agreed on the "
      "single-point energy to machine precision for every charge method, and all ran stable "
      "NPT dynamics (trajectories in traj_07/).")

charge method                  pathway                      T (K)    density (g/cm^3)     parity
------------------------------------------------------------------------------------------------
NAGLCharges (force field)      interchange         302.7 +/-  6.5      1.025 +/- 0.008    1.3e-07
                               openmmforcefields   301.1 +/-  5.6      1.024 +/- 0.008           
ToolkitAM1BCC (force field)    interchange         300.8 +/-  6.0      1.025 +/- 0.008    4.8e-07
                               openmmforcefields   299.8 +/-  5.6      1.025 +/- 0.009           
user charges (preset AM1-BCC)  interchange         301.2 +/-  5.9      1.025 +/- 0.009    1.6e-07
                               openmmforcefields   301.3 +/-  5.6      1.025 +/- 0.007           

Notebook 07 complete: one box of all of notebook 06's diverse solutes — two copies each, scattered apart, one copy atom-reversed, with unique nonzero inline vsite parameters — was parameterized through both Interchang